In [3]:
!curl -O https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed

  0      0   0      0   0      0      0      0                              0
100  1.06M 100  1.06M   0      0  1.94M      0                              0
100  1.06M 100  1.06M   0      0  1.94M      0                              0
100  1.06M 100  1.06M   0      0  1.94M      0                              0


In [4]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [5]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [10]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string


In [11]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [12]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size + 1]


tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [16]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"input: {context} the target is: {target}")

input: tensor([18]) the target is: 47
input: tensor([18, 47]) the target is: 56
input: tensor([18, 47, 56]) the target is: 57
input: tensor([18, 47, 56, 57]) the target is: 58
input: tensor([18, 47, 56, 57, 58]) the target is: 1
input: tensor([18, 47, 56, 57, 58,  1]) the target is: 15
input: tensor([18, 47, 56, 57, 58,  1, 15]) the target is: 47
input: tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is: 58


In [17]:
torch.manual_seed(2137)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb, yb = get_batch('train')
print(f'inputs: {xb.shape}')
print(xb)
print(f'targets: {yb.shape}')
print(yb)
print("\n\n")
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, : t+1]
        target = yb[b,t]
        print(f"input: {context} the target is: {target}")


inputs: torch.Size([4, 8])
tensor([[47, 56,  1, 54, 56, 53, 54, 53],
        [42,  8,  0,  0, 30, 27, 25, 17],
        [ 1, 47, 52,  1, 58, 46, 43,  1],
        [ 1, 54, 43, 53, 54, 50, 43,  6]])
targets: torch.Size([4, 8])
tensor([[56,  1, 54, 56, 53, 54, 53, 56],
        [ 8,  0,  0, 30, 27, 25, 17, 27],
        [47, 52,  1, 58, 46, 43,  1, 53],
        [54, 43, 53, 54, 50, 43,  6,  1]])



input: tensor([47]) the target is: 56
input: tensor([47, 56]) the target is: 1
input: tensor([47, 56,  1]) the target is: 54
input: tensor([47, 56,  1, 54]) the target is: 56
input: tensor([47, 56,  1, 54, 56]) the target is: 53
input: tensor([47, 56,  1, 54, 56, 53]) the target is: 54
input: tensor([47, 56,  1, 54, 56, 53, 54]) the target is: 53
input: tensor([47, 56,  1, 54, 56, 53, 54, 53]) the target is: 56
input: tensor([42]) the target is: 8
input: tensor([42,  8]) the target is: 0
input: tensor([42,  8,  0]) the target is: 0
input: tensor([42,  8,  0,  0]) the target is: 30
input: tensor([4